# Python 异常处理完整指南

> 涵盖 `try/except/else/finally` 结构、自定义异常类、上下文管理器（with 语句）


## 一、try/except/else/finally 结构

### 1. 完整结构图

```
try:
    ├── 尝试执行的代码
    └── 如果出错，立即跳转到 except
except ExceptionType1:
    └── 捕获特定异常，进行处理
except ExceptionType2 as e:
    └── 捕获另一种异常，e 是异常对象
except (TypeA, TypeB):
    └── 同时捕获多种异常
except:
    └── 捕获所有异常（不推荐，会掩盖 bug）
else:
    └── try 块没有异常时才执行
finally:
    └── 无论是否异常，最终都会执行
```


### 2. 代码示例：除法计算器

In [ ]:
def divide(a, b):
    try:
        print(f"尝试计算: {a} / {b}")
        result = a / b  # 可能抛出 ZeroDivisionError 或 TypeError

    except ZeroDivisionError as e:
        # 捕获除零错误
        print(f"除零错误: {e}")
        return None

    except TypeError as e:
        # 捕获类型错误（如传入字符串）
        print(f"类型错误: {e}")
        return None

    except Exception as e:
        # 兜底：捕获其他所有异常（谨慎使用）
        print(f"未知错误: {e}")
        return None

    else:
        # 只有 try 成功执行后才运行
        print("计算成功！")
        return result

    finally:
        # 无论成功与否，都会执行（常用于清理资源）
        print("--- 计算结束 ---")

# 测试
print("=== 正常情况 ===")
print(divide(10, 2))
print()

print("=== 除零错误 ===")
print(divide(10, 0))
print()

print("=== 类型错误 ===")
print(divide("10", 2))

### 3. 执行顺序总结

| 场景 | try | except | else | finally |
|------|-----|--------|------|---------|
| 无异常 | ✅ 执行 | ❌ 跳过 | ✅ 执行 | ✅ 执行 |
| 有匹配异常 | ✅ 执行到出错行 | ✅ 执行 | ❌ 跳过 | ✅ 执行 |
| 有无匹配异常 | ✅ 执行到出错行 | ❌ 跳过 | ❌ 跳过 | ✅ 执行后抛出 |

### 4. 异常捕获的优先级

**错误做法**：`Exception` 会捕获所有异常，后面的捕获不到。
**正确做法**：从具体到通用，像 `if-elif` 一样。

In [ ]:
# ❌ 错误：Exception 会捕获所有异常，后面的捕获不到
try:
    x = int("abc")
except Exception as e:      # ← 先捕获了所有异常
    print(f"通用异常: {e}")
except ValueError as e:     # ← 永远不会执行！
    print(f"值错误: {e}")

# ✅ 正确：从具体到通用
try:
    x = int("abc")
except ValueError as e:     # ← 先捕获具体的
    print(f"值错误: {e}")
except Exception as e:      # ← 最后兜底
    print(f"通用异常: {e}")

## 二、自定义异常类

### 1. 为什么要自定义？

> 内置异常（如 `ValueError`）语义太泛，自定义异常能**精确表达业务错误**，让代码更具可读性。

### 2. 继承关系

```
BaseException          ← 所有异常的基类（包括系统退出）
├── SystemExit         ← sys.exit() 触发
├── KeyboardInterrupt  ← Ctrl+C 触发
└── Exception          ← 所有普通异常的基类
    ├── ValueError
    ├── TypeError
    ├── KeyError
    └── ...（你的自定义异常在这里）
```


### 3. 基础版本：直接继承 Exception

In [ ]:
class BusinessError(Exception):
    """业务逻辑错误"""
    pass

# 使用
try:
    raise BusinessError("订单状态异常")
except BusinessError as e:
    print(f"捕获业务错误: {e}")

### 4. 进阶版本：添加额外信息

In [ ]:
class ValidationError(Exception):
    """参数验证失败"""

    def __init__(self, field, message):
        self.field = field
        self.message = message
        super().__init__(f"字段 '{field}' 验证失败: {message}")

# 使用
try:
    raise ValidationError("email", "格式不正确")
except ValidationError as e:
    print(f"字段: {e.field}")
    print(f"消息: {e.message}")
    print(f"完整信息: {e}")

### 5. 完整版本：错误码 + 层级体系

In [ ]:
class AppException(Exception):
    """应用异常基类"""
    code = 1000
    message = "应用错误"

    def __init__(self, detail=None):
        self.detail = detail or self.message
        super().__init__(self.detail)

    def to_dict(self):
        return {
            "code": self.code,
            "message": self.message,
            "detail": self.detail
        }

class NotFoundError(AppException):
    """资源不存在"""
    code = 404
    message = "资源未找到"

class PermissionDenied(AppException):
    """权限不足"""
    code = 403
    message = "无权访问"

class RateLimitExceeded(AppException):
    """请求频率超限"""
    code = 429
    message = "请求过于频繁"

# 使用
try:
    raise NotFoundError("用户 'alice' 不存在")
except AppException as e:
    print(f"错误码: {e.code}")
    print(f"错误信息: {e.to_dict()}")

### 6. 实战：用户服务

In [ ]:
class UserService:
    users = {"alice": {"age": 25}, "bob": {"age": 30}}

    def get_user(self, username):
        if username not in self.users:
            raise NotFoundError(f"用户 '{username}' 不存在")
        return self.users[username]

    def check_age(self, username, min_age):
        user = self.get_user(username)
        if user["age"] < min_age:
            raise ValidationError("age", f"年龄需大于 {min_age}")
        return True

# 测试
service = UserService()

print("=== 正常查询 ===")
print(service.get_user("alice"))

print()
print("=== 用户不存在 ===")
try:
    service.get_user("charlie")
except NotFoundError as e:
    print(f"[{e.code}] {e.message}: {e.detail}")

print()
print("=== 年龄验证失败 ===")
try:
    service.check_age("alice", 30)  # alice 25岁，不满足
except ValidationError as e:
    print(f"验证失败: {e.field} - {e}")
except AppException as e:
    print(f"[{e.code}] {e.to_dict()}")

### 7. 异常链：保留原始异常

In [ ]:
class DatabaseError(AppException):
    """数据库操作失败"""
    code = 500
    message = "数据库操作失败"

def fetch_data():
    try:
        # 模拟数据库操作
        import sqlite3
        conn = sqlite3.connect("nonexistent.db")
    except sqlite3.Error as e:
        # 抛出自定义异常，但保留原始异常信息
        raise DatabaseError("无法连接数据库") from e

try:
    fetch_data()
except DatabaseError as e:
    print(f"业务错误: {e}")
    print(f"原始错误: {e.__cause__}")  # 通过 __cause__ 访问原始异常

## 三、上下文管理器（with 语句）

### 1. 为什么要用 with？

> **自动管理资源的获取和释放**，避免忘记关闭文件、锁、连接等。

```python
# ❌ 不用 with：容易忘记关闭，异常时也不会关闭
f = open("data.txt", "r")
data = f.read()
f.close()  # 如果上面出错了，这行不会执行！

# ✅ 用 with：自动关闭，即使发生异常
with open("data.txt", "r") as f:
    data = f.read()
# ← 这里自动调用 f.close()
```

### 2. with 的本质：上下文管理器协议

一个对象要成为上下文管理器，必须实现两个方法：

| 方法 | 时机 | 作用 |
|------|------|------|
| `__enter__()` | 进入 with 块时 | 获取资源，返回值赋给 `as` 变量 |
| `__exit__(exc_type, exc_val, exc_tb)` | 离开 with 块时 | 释放资源，处理异常 |

```
with EXPR as VAR:
    BLOCK

# 等价于：
manager = (EXPR)
VAR = manager.__enter__()
try:
    BLOCK
finally:
    manager.__exit__(异常类型, 异常值, 回溯)
```


### 3. 自定义上下文管理器：类实现

In [ ]:
import time

class Timer:
    """计时器上下文管理器"""

    def __enter__(self):
        self.start = time.time()
        print("⏱️ 开始计时...")
        return self  # 返回自己，as 变量能访问

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.end = time.time()
        print(f"⏹️ 结束计时，耗时: {self.end - self.start:.4f} 秒")

        # 返回 True 会"吞掉"异常，with 块外看不到
        # 返回 False（或不返回）异常会继续向上抛
        return False

# 使用
with Timer() as t:
    time.sleep(1)
    print("执行一些操作...")

### 4. 自定义上下文管理器：装饰器实现（更简洁）

In [ ]:
from contextlib import contextmanager

@contextmanager
def managed_resource(name):
    """用生成器实现上下文管理器"""
    print(f"获取资源: {name}")
    resource = {"name": name, "status": "active"}

    try:
        yield resource  # ← 这里返回给 as 变量
    except Exception as e:
        print(f"发生异常: {e}")
        raise  # 重新抛出，不吞掉异常
    finally:
        print(f"释放资源: {name}")

# 使用
with managed_resource("数据库连接") as res:
    print(f"使用资源: {res}")

### 5. 实战案例 1：数据库事务

In [ ]:
from contextlib import contextmanager

@contextmanager
def transaction(db_connection):
    """自动提交/回滚的事务上下文"""
    cursor = db_connection.cursor()
    try:
        yield cursor
        db_connection.commit()
        print("✅ 事务提交成功")
    except Exception as e:
        db_connection.rollback()
        print(f"❌ 事务回滚: {e}")
        raise

# 使用
import sqlite3
conn = sqlite3.connect(":memory:")

with transaction(conn) as cursor:
    cursor.execute("CREATE TABLE users (id INTEGER PRIMARY KEY, name TEXT)")
    cursor.execute("INSERT INTO users (name) VALUES ('Alice')")
    # 如果这里出错，自动回滚

# 验证
result = conn.execute("SELECT * FROM users").fetchall()
print(f"查询结果: {result}")

### 6. 实战案例 2：重试机制

In [4]:
import time

class RetryContext:
    """重试上下文管理器（类实现，支持多次尝试）"""
    
    def __init__(self, max_attempts=3, delay=1, exceptions=(Exception,)):
        self.max_attempts = max_attempts
        self.delay = delay
        self.exceptions = exceptions
    
    def __enter__(self):
        self.attempt = 0
        return self  # 返回 self，as 变量可以访问属性
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        # __exit__ 在每次 with 块结束时调用
        # 但我们的重试逻辑不在这里，而是在装饰器/函数里
        pass
    
    def run(self, func, *args, **kwargs):
        """执行带重试的函数"""
        for attempt in range(1, self.max_attempts + 1):
            try:
                self.attempt = attempt
                return func(*args, **kwargs)
            except self.exceptions as e:
                print(f"第 {attempt} 次尝试失败: {e}")
                if attempt == self.max_attempts:
                    raise
                time.sleep(self.delay)


# ========== 更简单的方式：直接用函数装饰器 ==========

def retry(max_attempts=3, delay=1, exceptions=(Exception,)):
    """函数装饰器版重试"""
    def decorator(func):
        def wrapper(*args, **kwargs):
            for attempt in range(1, max_attempts + 1):
                try:
                    return func(*args, **kwargs)
                except exceptions as e:
                    print(f"第 {attempt} 次尝试失败: {e}")
                    if attempt == max_attempts:
                        raise
                    time.sleep(delay)
        return wrapper
    return decorator


# ========== 使用 ==========

import random

@retry(max_attempts=3, delay=0.5, exceptions=(ConnectionError,))
def unstable_operation():
    if random.random() < 0.7:
        raise ConnectionError("网络波动")
    return "成功！"

random.seed(42)
try:
    result = unstable_operation()
    print(f"最终结果: {result}")
except ConnectionError as e:
    print(f"最终失败: {e}")

第 1 次尝试失败: 网络波动
第 2 次尝试失败: 网络波动
第 3 次尝试失败: 网络波动
最终失败: 网络波动


### 7. 实战案例 3：临时修改环境变量

In [ ]:
import os
from contextlib import contextmanager

@contextmanager
def temp_env_var(key, value):
    """临时修改环境变量"""
    old_value = os.environ.get(key)
    os.environ[key] = value
    try:
        yield
    finally:
        if old_value is None:
            del os.environ[key]
        else:
            os.environ[key] = old_value

# 使用
print(f"修改前 DEBUG: {os.environ.get('DEBUG')}")

with temp_env_var("DEBUG", "1"):
    print(f"临时值: {os.environ['DEBUG']}")

print(f"恢复后 DEBUG: {os.environ.get('DEBUG')}")

### 8. 多个 with 嵌套

In [ ]:
# 方式一：嵌套
# with open("input.txt") as fin:
#     with open("output.txt", "w") as fout:
#         fout.write(fin.read().upper())

# 方式二：一行并列（Python 3.10+ 推荐）
# with (
#     open("input.txt") as fin,
#     open("output.txt", "w") as fout
# ):
#     fout.write(fin.read().upper())

# 演示用 StringIO 模拟
from io import StringIO

input_data = StringIO("hello world")
output_data = StringIO()

with input_data as fin, output_data as fout:
    fout.write(fin.read().upper())

print(f"输出结果: {output_data.getvalue()}")

## 四、三者结合：完整的异常处理体系

In [ ]:
from contextlib import contextmanager

# 1. 自定义异常
class APIError(Exception):
    def __init__(self, status_code, message):
        self.status_code = status_code
        self.message = message
        super().__init__(f"[{status_code}] {message}")

# 2. 上下文管理器
@contextmanager
def api_client(base_url):
    """模拟 API 客户端，自动管理会话"""
    print(f"🔗 连接到 {base_url}")
    session = {"connected": True, "url": base_url}

    try:
        yield session
    except APIError as e:
        print(f"🚨 API 错误处理: {e}")
        raise
    finally:
        session["connected"] = False
        print("🔌 断开连接")

# 3. 使用 try/except/else/finally
def fetch_user(user_id):
    with api_client("https://api.example.com") as client:
        try:
            print(f"请求用户 {user_id}...")
            if user_id < 0:
                raise APIError(400, "无效的用户ID")
            if user_id > 1000:
                raise APIError(404, "用户不存在")

            result = {"id": user_id, "name": f"User_{user_id}"}

        except APIError as e:
            print(f"捕获到 APIError: {e.status_code}")
            return {"error": e.message}
        else:
            print("✅ 请求成功")
            return result
        finally:
            print("📝 记录请求日志")

# 测试
print("=== 正常请求 ===")
print(fetch_user(42))
print()

print("=== 400 错误 ===")
print(fetch_user(-1))
print()

print("=== 404 错误 ===")
print(fetch_user(9999))

## 五、知识图谱总结

```
异常处理体系
├── try/except/else/finally
│   ├── try: 尝试执行
│   ├── except: 捕获并处理（从具体到通用）
│   ├── else: 无异常时执行
│   └── finally: 无论是否异常都执行
│
├── 自定义异常
│   ├── 继承 Exception 或其子类
│   ├── 添加业务语义（错误码、字段等）
│   ├── 异常链：raise ... from ...
│   └── 层级体系：基类 → 具体异常
│
└── 上下文管理器 (with)
    ├── 协议：__enter__ + __exit__
    ├── 方式一：类实现
    ├── 方式二：@contextmanager 装饰器（推荐）
    └── 用途：文件、锁、事务、临时状态
```

### 关键要点速查

| 主题 | 核心要点 |
|------|---------|
| **try/except** | 从具体到通用捕获，避免裸 `except:` |
| **else** | 无异常时执行，分离正常逻辑 |
| **finally** | 资源清理，必定执行 |
| **自定义异常** | 继承 `Exception`，添加业务语义 |
| **异常链** | `raise NewError(...) from e` 保留原始异常 |
| **with** | 自动管理资源，实现 `__enter__` + `__exit__` |
| **@contextmanager** | 用生成器简化上下文管理器编写 |
